# 面试问题：怎样为 70B 训练任务规划 DP、TP、PP 和 Sequence Parallel，并尊重节点拓扑？

        ## 可直接复述的回答主线

        1. 3D 并行规划首先是显存和拓扑约束问题，其次才是吞吐调优问题。
2. 数据并行复制模型但扩大全局批次，张量并行切层内矩阵，流水并行切层，序列并行切激活。
3. 朴素数据并行会在每张卡复制 70B 模型状态，即使 GPU 总显存很多仍然单卡溢出。
4. 候选计划必须满足 DP 乘 TP 乘 PP 等于设备数，并显式估算模型状态、激活、通信和流水气泡。
5. TP 最好限制在高速互联域内，跨节点 TP 的延迟往往抵消更低的单卡显存。
6. 最终方案要用真实 profiler 校准通信重叠和激活峰值，公式只能做第一轮剪枝。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是一项 70B 中文模型继续预训练任务，集群为两台节点、每台 8 张 80GB GPU。六个候选计划保留 DP、TP、PP、SP、微批数和节点拓扑字段，属于按真实规划表结构构造的脱敏离线样本。

In [1]:
import math  # 引入计划合法性与有限数检查所需的数学函数。
cluster = {"nodes": 2, "gpus_per_node": 8, "gpu_memory_gb": 80.0, "total_gpus": 16}  # 定义两节点训练集群的容量和拓扑边界。
workload = {"params_b": 70.0, "activation_gb": 24.0, "microbatches": 8, "hidden": 8192}  # 定义 70B 训练负载的关键规划字段。
plans = [{"name": "纯DP", "dp": 16, "tp": 1, "pp": 1, "sp": 1}, {"name": "DP8-TP2", "dp": 8, "tp": 2, "pp": 1, "sp": 2}, {"name": "DP4-TP4", "dp": 4, "tp": 4, "pp": 1, "sp": 4}, {"name": "DP4-TP2-PP2", "dp": 4, "tp": 2, "pp": 2, "sp": 2}, {"name": "DP2-TP4-PP2", "dp": 2, "tp": 4, "pp": 2, "sp": 4}, {"name": "TP8-PP2", "dp": 1, "tp": 8, "pp": 2, "sp": 8}, {"name": "跨节点TP16", "dp": 1, "tp": 16, "pp": 1, "sp": 16}]  # 给出七个覆盖常见切分方向的候选。
print("教学实验输入：70B 模型，两节点各 8 卡")  # 输出本次并行规划的业务背景。
print("方案             DP  TP  PP  SP  设备乘积")  # 输出候选计划表头。
for plan in plans:  # 逐条展示计划维度和设备乘积。
    product = plan["dp"] * plan["tp"] * plan["pp"]  # 计算当前计划占用的设备总数。
    print(f"{plan['name']:<16} {plan['dp']:>2} {plan['tp']:>3} {plan['pp']:>3} {plan['sp']:>3} {product:>8}")  # 输出当前候选的并行配置。

教学实验输入：70B 模型，两节点各 8 卡
方案             DP  TP  PP  SP  设备乘积
纯DP              16   1   1   1       16
DP8-TP2           8   2   1   2       16
DP4-TP4           4   4   1   4       16
DP4-TP2-PP2       4   2   2   2       16
DP2-TP4-PP2       2   4   2   4       16
TP8-PP2           1   8   2   8       16
跨节点TP16           1  16   1  16       16


## 2. Baseline / 基线：只做数据并行

纯 DP 实现最简单，但参数、梯度和大部分优化器状态在每张卡上复制。下面先显示它为何在 80GB 单卡上不可行。

In [2]:
def memory_breakdown(plan):  # 分解当前并行计划的单卡模型状态与激活峰值。
    model_partition = plan["tp"] * plan["pp"]  # 计算参数和梯度被切成多少模型分片。
    parameter_and_gradient_gb = workload["params_b"] * 4.0 / model_partition  # 按 BF16 参数加梯度估算单卡复制量。
    optimizer_gb = workload["params_b"] * 8.0 / (model_partition * plan["dp"])  # 假设优化器状态在 DP 组内分片。
    activation_gb = workload["activation_gb"] / (plan["tp"] * plan["pp"] * plan["sp"])  # 估算 TP、PP、SP 对激活峰值的共同削减。
    total_gb = parameter_and_gradient_gb + optimizer_gb + activation_gb  # 汇总单卡模型状态和激活占用。
    return parameter_and_gradient_gb, optimizer_gb, activation_gb, total_gb  # 返回可审计的显存分项。
baseline = plans[0]  # 选择纯数据并行作为朴素基线。
baseline_memory = memory_breakdown(baseline)  # 计算纯 DP 的单卡显存分项。
print("Baseline 单卡显存分解")  # 标记当前输出属于纯 DP 基线。
print(f"参数+梯度={baseline_memory[0]:.1f}GB，优化器={baseline_memory[1]:.1f}GB，激活={baseline_memory[2]:.1f}GB")  # 输出三项来源。
print(f"总计={baseline_memory[3]:.1f}GB，超过80GB={baseline_memory[3] - cluster['gpu_memory_gb']:.1f}GB")  # 显示基线 OOM 的量级。

Baseline 单卡显存分解
参数+梯度=280.0GB，优化器=35.0GB，激活=24.0GB
总计=339.0GB，超过80GB=259.0GB


## 3. 底层实现：同时估算显存、通信与流水气泡

通信量是教学近似：TP 使用层内激活通信，DP 使用梯度归约，PP 由微批数决定气泡。我们保留每个分项，避免用一个神秘总分替代机制解释。

In [3]:
def communication_and_bubble(plan):  # 估算当前计划的通信代理量和流水气泡比例。
    model_partition = plan["tp"] * plan["pp"]  # 复用模型切分数统一通信口径。
    tp_collective_gb = 6.0 * (plan["tp"] - 1) / max(plan["tp"], 1)  # 用激活代理量估算 TP 集合通信。
    dp_reduce_gb = workload["params_b"] * 2.0 / model_partition * (plan["dp"] - 1) / max(plan["dp"], 1)  # 估算 DP 梯度归约字节。
    bubble = (plan["pp"] - 1) / (workload["microbatches"] + plan["pp"] - 1)  # 计算一前一后近似流水气泡。
    return tp_collective_gb, dp_reduce_gb, bubble  # 返回通信和气泡三个独立诊断量。
evaluated = []  # 收集所有合法设备乘积候选的规划结果。
for plan in plans:  # 逐个评估并行计划。
    legal_product = plan["dp"] * plan["tp"] * plan["pp"] == cluster["total_gpus"]  # 验证并行维度是否覆盖全部设备。
    memory = memory_breakdown(plan)  # 计算当前计划的显存分项。
    tp_gb, dp_gb, bubble = communication_and_bubble(plan)  # 计算当前计划的通信和气泡。
    topology_ok = plan["tp"] <= cluster["gpus_per_node"]  # 检查 TP 是否留在单节点高速互联域内。
    evaluated.append({"plan": plan, "legal": legal_product, "memory": memory, "tp_gb": tp_gb, "dp_gb": dp_gb, "bubble": bubble, "topology_ok": topology_ok})  # 保存完整诊断而非只有是否可行。
print("方案             显存GB  TP通信  DP通信  气泡%  拓扑安全")  # 输出核心规划分项表头。
for row in evaluated:  # 逐条展示显存与通信权衡。
    print(f"{row['plan']['name']:<16} {row['memory'][3]:>7.1f} {row['tp_gb']:>7.1f} {row['dp_gb']:>7.1f} {100 * row['bubble']:>6.1f} {str(row['topology_ok']):>9}")  # 输出当前计划的底层估算。

方案             显存GB  TP通信  DP通信  气泡%  拓扑安全
纯DP                339.0     0.0   131.2    0.0      True
DP8-TP2            181.0     3.0    61.2    0.0      True
DP4-TP4            106.5     4.5    26.2    0.0      True
DP4-TP2-PP2        108.0     3.0    26.2   11.1      True
DP2-TP4-PP2         70.8     4.5     8.8   11.1      True
TP8-PP2             52.7     5.2     0.0   11.1      True
跨节点TP16             52.6     5.6     0.0    0.0     False


## 4. 结果表与结果解读

先过滤设备乘积和显存，再对拓扑安全候选按通信代理量加气泡惩罚排序。结果不是说 PP 永远更好，而是说明在此集群上纯 DP 不可行、跨节点 TP 风险高。

In [4]:
feasible = [row for row in evaluated if row["legal"] and row["memory"][3] <= cluster["gpu_memory_gb"] and row["topology_ok"]]  # 同时应用设备、显存和拓扑门禁。
for row in feasible:  # 为每个可行候选计算可解释排序分数。
    row["score"] = row["tp_gb"] + row["dp_gb"] + 100.0 * row["bubble"]  # 合并通信代理量和气泡百分比。
selected = min(feasible, key=lambda row: row["score"])  # 选择当前教学模型下代价最低的安全计划。
print("通过硬约束的候选排序")  # 标记下表已经排除 OOM 和跨节点 TP。
print("方案             综合代理分   单卡显存GB")  # 输出排序表头。
for row in sorted(feasible, key=lambda item: item["score"]):  # 按统一分数展示全部安全候选。
    print(f"{row['plan']['name']:<16} {row['score']:>10.2f} {row['memory'][3]:>11.1f}")  # 输出排序分数与显存余量。
print(f"解读：当前首选 {selected['plan']['name']}，TP={selected['plan']['tp']} 留在单节点内，PP={selected['plan']['pp']} 以可控气泡换显存。")  # 直接解释最终选择原因。

通过硬约束的候选排序
方案             综合代理分   单卡显存GB
TP8-PP2               16.36        52.7
DP2-TP4-PP2           24.36        70.8
解读：当前首选 TP8-PP2，TP=8 留在单节点内，PP=2 以可控气泡换显存。


## 5. 失败案例与修正

如果只最小化单卡显存，TP16 看起来很诱人，但它跨越两台节点，层内 collective 会走较慢链路。修正方式是把 TP 通信域作为硬门禁，而不是事后靠平均吞吐掩盖长尾。

In [5]:
memory_only = min([row for row in evaluated if row["legal"] and row["memory"][3] <= cluster["gpu_memory_gb"]], key=lambda row: row["memory"][3])  # 模拟只追求最低单卡显存的错误策略。
cross_node_hops = math.ceil(memory_only["plan"]["tp"] / cluster["gpus_per_node"])  # 估算该 TP 组跨越的节点数量。
safe_choice = selected["plan"]  # 读取加入拓扑门禁后的安全计划。
print(f"错误行为：{memory_only['plan']['name']} 单卡仅 {memory_only['memory'][3]:.1f}GB，但 TP 组跨 {cross_node_hops} 个节点。")  # 展示最低显存策略隐藏的通信风险。
print(f"修正行为：{safe_choice['name']} 将 TP={safe_choice['tp']} 限制在每节点 {cluster['gpus_per_node']} 卡以内。")  # 展示拓扑感知选择。

错误行为：跨节点TP16 单卡仅 52.6GB，但 TP 组跨 2 个节点。
修正行为：TP8-PP2 将 TP=8 限制在每节点 8 卡以内。


## 6. 生产边界

生产规划还要纳入实际网络带宽、collective 拓扑、激活重计算、MoE 专家、序列长度分布、梯度累积和框架保留显存。最关键的替换点是用 profiler 峰值和 step time 校准这里的代理公式。

In [6]:
microbatch_schedule = [(microbatch, stage, microbatch + stage) for microbatch in range(3) for stage in range(selected["plan"]["pp"])]  # 构造前三个微批经过流水 stage 的简化时间槽。
print("首选计划的前几个流水事件：", microbatch_schedule)  # 输出调度顺序帮助理解 PP 气泡从何而来。

首选计划的前几个流水事件： [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 2), (2, 0, 2), (2, 1, 3)]


## 7. 最小回归测试

回归测试只保护设备乘积、单卡显存、拓扑域和确定性选择。

In [7]:
assert len(plans) >= 5  # 保证规划案例具有足够多的候选对照。
assert selected["plan"]["dp"] * selected["plan"]["tp"] * selected["plan"]["pp"] == cluster["total_gpus"]  # 保证首选计划使用全部设备。
assert selected["memory"][3] <= cluster["gpu_memory_gb"]  # 保证首选计划不会在教学估算下单卡 OOM。
assert selected["plan"]["tp"] <= cluster["gpus_per_node"]  # 保证 TP 集合通信不跨节点。
assert baseline_memory[3] > cluster["gpu_memory_gb"]  # 保证纯 DP 基线确实构成可解释失败。